In [43]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

In [44]:
data = pd.read_csv('train.csv')

we vizualize the first 5 training exemples "images" with the head function , each image is a represented by a pixel raw

In [45]:
data.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [46]:
data = np.array(data)
m,n = data.shape
np.random.shuffle(data)

data_test = data[0:1000].T 
Y_test = data_test[0]
X_test = data_test[1:n]

data_train = data[1000:m].T
Y_train= data_train[0]
X_train = data_train[1:n]



X_train = X_train / 255.0
X_test = X_test / 255.0


# 2-Layer Neural Network Architecture (MNIST)

Our neural network has a simple two-layer architecture:
- **Input layer ($A^{[0]}$):** 784 units corresponding to the 784 pixels in each $28 \times 28$ input image.
- **Hidden layer ($A^{[1]}$):** 10 units with **ReLU** activation.
- **Output layer ($A^{[2]}$):** 10 units corresponding to the 10 digit classes with **Softmax** activation.

---

## 1. Forward Propagation
$$Z^{[1]} = W^{[1]}X + b^{[1]}$$
$$A^{[1]} = g_{\text{ReLU}}(Z^{[1]})$$
$$Z^{[2]} = W^{[2]}A^{[1]} + b^{[2]}$$
$$A^{[2]} = g_{\text{softmax}}(Z^{[2]})$$

---

## 2. Backward Propagation
$$dZ^{[2]} = A^{[2]} - Y$$
$$dW^{[2]} = \frac{1}{m} dZ^{[2]} A^{[1]T}$$
$$dB^{[2]} = \frac{1}{m} \sum \text{axis=1, keepdims=True}(dZ^{[2]})$$
$$dZ^{[1]} = W^{[2]T} dZ^{[2]} * g^{[1]\prime}(Z^{[1]})$$
$$dW^{[1]} = \frac{1}{m} dZ^{[1]} A^{[0]T}$$
$$dB^{[1]} = \frac{1}{m} \sum \text{axis=1, keepdims=True}(dZ^{[1]})$$

---

## 3. Parameter Updates
$$W^{[2]} := W^{[2]} - \alpha dW^{[2]}$$
$$b^{[2]} := b^{[2]} - \alpha dB^{[2]}$$
$$W^{[1]} := W^{[1]} - \alpha dW^{[1]}$$
$$b^{[1]} := b^{[1]} - \alpha dB^{[1]}$$

---

## 4. Variables and Shapes ($m$ = number of examples)

### Forward Propagation
- $A^{[0]} = X$: $784 \times m$
- $Z^{[1]} \sim A^{[1]}$: $10 \times m$
- $W^{[1]}$: $10 \times 784$
- $b^{[1]}$: $10 \times 1$
- $Z^{[2]} \sim A^{[2]}$: $10 \times m$
- $W^{[2]}$: $10 \times 10$
- $b^{[2]}$: $10 \times 1$

### Backward Propagation
- $dZ^{[2]}$: $10 \times m$
- $dW^{[2]}$: $10 \times 10$
- $dB^{[2]}$: $10 \times 1$
- $dZ^{[1]}$: $10 \times m$
- $dW^{[1]}$: $10 \times 784$
- $dB^{[1]}$: $10 \times 1$

In [47]:


def init_params():
    
    W1 = np.random.rand(10, 784) - 0.5
    b1 = np.random.rand(10, 1) - 0.5
    W2 = np.random.rand(10, 10) - 0.5
    b2 = np.random.rand(10, 1) - 0.5
    return W1, b1, W2, b2

def ReLU(Z):
    return np.maximum(Z, 0)

def softmax(Z):
  
    exp_Z = np.exp(Z - np.max(Z, axis=0, keepdims=True))
    return exp_Z / np.sum(exp_Z, axis=0, keepdims=True)
    
def forward_prop(W1, b1, W2, b2, X):
    Z1 = W1.dot(X) + b1
    A1 = ReLU(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = softmax(Z2)
    return Z1, A1, Z2, A2

def ReLU_deriv(Z):
    return Z > 0

def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, Y.max() + 1))
    one_hot_Y[np.arange(Y.size), Y] = 1
    return one_hot_Y.T

def backward_prop(Z1, A1, Z2, A2, W1, W2, X, Y):
    m = X.shape[1] 
    one_hot_Y = one_hot(Y)
    
    dZ2 = A2 - one_hot_Y
    dW2 = (1 / m) * dZ2.dot(A1.T)
    
 
    db2 = (1 / m) * np.sum(dZ2, axis=1, keepdims=True)
    
    dZ1 = W2.T.dot(dZ2) * ReLU_deriv(Z1)
    dW1 = (1 / m) * dZ1.dot(X.T)
    db1 = (1 / m) * np.sum(dZ1, axis=1, keepdims=True)
    
    return dW1, db1, dW2, db2

def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1    
    W2 = W2 - alpha * dW2  
    b2 = b2 - alpha * db2    
    return W1, b1, W2, b2

In [48]:
def get_predictions(A2):
    return np.argmax(A2, 0)

def get_accuracy(predictions, Y):
    print(predictions, Y)
    return np.sum(predictions == Y) / Y.size

def gradient_descent(X, Y, alpha, iterations):
    W1, b1, W2, b2 = init_params()
    for i in range(iterations):
        Z1, A1, Z2, A2 = forward_prop(W1, b1, W2, b2, X)
        dW1, db1, dW2, db2 = backward_prop(Z1, A1, Z2, A2, W1, W2, X, Y)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)
        if i % 10 == 0:
            print("Iteration: ", i)
            predictions = get_predictions(A2)
            print(get_accuracy(predictions, Y))
    return W1, b1, W2, b2


In [49]:

W1, b1, W2, b2 = gradient_descent(X_train, Y_train, alpha=0.1, iterations=500)

Iteration:  0
[3 7 1 ... 7 7 0] [1 9 5 ... 1 0 4]
0.08907317073170731
Iteration:  10
[7 7 1 ... 7 7 5] [1 9 5 ... 1 0 4]
0.16263414634146342
Iteration:  20
[7 4 1 ... 7 7 5] [1 9 5 ... 1 0 4]
0.22453658536585366
Iteration:  30
[7 4 0 ... 7 0 7] [1 9 5 ... 1 0 4]
0.28087804878048783
Iteration:  40
[3 4 0 ... 3 0 4] [1 9 5 ... 1 0 4]
0.33890243902439027
Iteration:  50
[1 4 0 ... 3 0 4] [1 9 5 ... 1 0 4]
0.40195121951219515
Iteration:  60
[1 4 5 ... 1 0 4] [1 9 5 ... 1 0 4]
0.4690243902439024
Iteration:  70
[1 4 5 ... 1 0 4] [1 9 5 ... 1 0 4]
0.5241707317073171
Iteration:  80
[1 4 5 ... 1 0 4] [1 9 5 ... 1 0 4]
0.5646097560975609
Iteration:  90
[1 4 5 ... 1 0 4] [1 9 5 ... 1 0 4]
0.5976585365853658
Iteration:  100
[1 4 5 ... 1 0 4] [1 9 5 ... 1 0 4]
0.6268048780487805
Iteration:  110
[1 4 5 ... 1 0 4] [1 9 5 ... 1 0 4]
0.6547804878048781
Iteration:  120
[1 4 5 ... 1 0 4] [1 9 5 ... 1 0 4]
0.6785853658536586
Iteration:  130
[1 9 5 ... 1 0 4] [1 9 5 ... 1 0 4]
0.6981951219512195
Iteration: 